# Feature Engineering

In this notebook, we will perform feature engineering on the customer dataset. This includes creating the RFM (Recency, Frequency, Monetary) table, handling outliers, scaling features, and saving the processed data.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Load the cleaned data
data = pd.read_csv('../data/processed/cleaned_customers.csv')

# Display the first few rows of the dataset
data.head()

## Creating the RFM Table

We will create the RFM table to segment customers based on their purchasing behavior.

In [ ]:
# Create RFM table
snapshot_date = data['InvoiceDate'].max() + pd.DateOffset(days=1)
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'count',
    'TotalSum': 'sum'
}).rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalSum': 'Monetary'
})

# Display the RFM table
rfm.head()

## Handling Outliers

Next, we will handle outliers in the Monetary value to ensure that our analysis is not skewed.

In [ ]:
# Identify outliers using IQR method
Q1 = rfm['Monetary'].quantile(0.25)
Q3 = rfm['Monetary'].quantile(0.75)
IQR = Q3 - Q1
rfm = rfm[~((rfm['Monetary'] < (Q1 - 1.5 * IQR)) | (rfm['Monetary'] > (Q3 + 1.5 * IQR)))]

# Display the RFM table after outlier removal
rfm.head()

## Scaling Features

We will scale the RFM features to prepare them for clustering.

In [ ]:
# Scale the RFM features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

# Convert scaled data back to DataFrame
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=rfm.columns)
rfm_scaled_df.head()

## Saving the Processed Data

Finally, we will save the processed RFM data for use in the modeling phase.

In [ ]:
# Save the processed RFM data
rfm_scaled_df.to_csv('../data/processed/rfm_scaled.csv', index=False)
print('Processed RFM data saved successfully!')